# Démonstration – ALNS pour le CVRPTW (instances de Solomon)

**Une seule cellule à exécuter** (Shift + Entrée). Les réglages (instances, nombre d'itérations, source locale ou GitHub) sont dans la section *PARAMÈTRES À MODIFIER*, juste sous les importations.

In [ ]:
# =====================================================================
# DÉMONSTRATION – ALNS + recuit simulé pour le CVRPTW (instances de Solomon)
# =====================================================================

# --- Installation automatique des librairies manquantes (rien n'est fait si tout est déjà installé)
import importlib
import importlib.util
import subprocess
import sys

for _module, _paquet in [("numpy", "numpy"), ("matplotlib", "matplotlib"), ("vrplib", "vrplib")]:
    if importlib.util.find_spec(_module) is None:
        print(f"Installation de {_paquet} ...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", _paquet])
        except Exception as err:
            raise RuntimeError(f"Impossible d'installer {_paquet} (internet ? Python >= 3.10 pour vrplib ?) : {err}")
        importlib.invalidate_caches()

# --- Importations
import math
import os
import random
import tempfile
import time
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import vrplib

# Les 56 instances de Solomon (C = clients groupés, R = aléatoires, RC = mixtes ;
# série 1 = fenêtres serrées / petits camions, série 2 = fenêtres larges / grands camions)
SOLOMON = ([f"C1{i:02d}" for i in range(1, 10)] + [f"C2{i:02d}" for i in range(1, 9)]
           + [f"R1{i:02d}" for i in range(1, 13)] + [f"R2{i:02d}" for i in range(1, 12)]
           + [f"RC1{i:02d}" for i in range(1, 9)] + [f"RC2{i:02d}" for i in range(1, 9)])
INSTANCES_A_LANCER = ["R101", "RC201"]
NB_ITERATIONS = 2000
GRAINE = 1
# "local"  : uniquement depuis le dossier DOSSIER_LOCAL ci-dessous
# "github" : uniquement depuis GitHub (aucun fichier local nécessaire, internet obligatoire)
# "auto"   : d'abord le dossier local, et si l'instance n'y est pas, GitHub
SOURCE_INSTANCES = "auto"
DOSSIER_LOCAL = os.path.join("instances", "solomon")
URL_GITHUB = "https://raw.githubusercontent.com/PyVRP/VRPLIB/main/tests/data/cvrplib/Vrp-Set-Solomon"
LANCER_TEST_UNITAIRE = True

# --- Vérification des paramètres avec messages d'erreurs
if isinstance(INSTANCES_A_LANCER, str):
    INSTANCES_A_LANCER = [INSTANCES_A_LANCER]
INSTANCES_A_LANCER = list(dict.fromkeys(str(n).strip().upper() for n in INSTANCES_A_LANCER))
if not INSTANCES_A_LANCER:
    raise ValueError("INSTANCES_A_LANCER est vide : indiquez au moins une instance, par exemple [\"R101\"].")
if not (isinstance(NB_ITERATIONS, int) and NB_ITERATIONS >= 1):
    raise ValueError("NB_ITERATIONS doit être un entier >= 1.")
if SOURCE_INSTANCES not in ("local", "github", "auto"):
    raise ValueError('SOURCE_INSTANCES doit valoir "local", "github" ou "auto".')
inconnues = [n for n in INSTANCES_A_LANCER if n not in SOLOMON]
if inconnues:
    print(f"Attention : {inconnues} ne fait pas partie des 56 instances de Solomon "
          f"(le programme essaiera quand même de les lire).")


# ========================================================================
# Chargement des instances (local et/ou GitHub)
# ========================================================================
def _lire_fichier_instance(chemin):
    """Lit un fichier d'instance : format Solomon (.txt) ou VRPLIB (.vrp). Essaie l'autre format si besoin."""
    formats = ["solomon", "vrplib"] if chemin.lower().endswith(".txt") else ["vrplib", "solomon"]
    derniere_erreur = None
    for fmt in formats:
        try:
            return vrplib.read_instance(chemin, instance_format=fmt)
        except Exception as err:
            derniere_erreur = err
    raise ValueError(f"Format d'instance non reconnu pour {chemin} : {derniere_erreur}")


def _charger_local(name):
    """Cherche NOM.vrp (ou NOM.txt) et NOM.sol dans DOSSIER_LOCAL."""
    sol = os.path.join(DOSSIER_LOCAL, f"{name}.sol")
    for ext in ("vrp", "txt"):
        inst = os.path.join(DOSSIER_LOCAL, f"{name}.{ext}")
        if os.path.exists(inst) and os.path.exists(sol):
            return _lire_fichier_instance(inst), vrplib.read_solution(sol)
    raise FileNotFoundError(f"{name}.vrp (ou .txt) et {name}.sol introuvables dans "
                            f"« {os.path.abspath(DOSSIER_LOCAL)} »")


def _telecharger(url):
    try:
        with urllib.request.urlopen(url, timeout=30) as reponse:
            return reponse.read()
    except Exception as err:
        raise ConnectionError(f"téléchargement impossible : {url} ({err})") from err


def _charger_github(name):
    """Télécharge NOM.txt et NOM.sol depuis GitHub, sans rien enregistrer dans votre dossier."""
    contenu_inst = _telecharger(f"{URL_GITHUB}/{name}.txt")
    contenu_sol = _telecharger(f"{URL_GITHUB}/{name}.sol")
    with tempfile.TemporaryDirectory() as tmp:
        p_inst, p_sol = os.path.join(tmp, f"{name}.txt"), os.path.join(tmp, f"{name}.sol")
        with open(p_inst, "wb") as f:
            f.write(contenu_inst)
        with open(p_sol, "wb") as f:
            f.write(contenu_sol)
        return _lire_fichier_instance(p_inst), vrplib.read_solution(p_sol)


def load_instance(name):
    """Renvoie (instance, meilleure solution connue) selon SOURCE_INSTANCES."""
    if SOURCE_INSTANCES == "local":
        return _charger_local(name)
    if SOURCE_INSTANCES == "github":
        return _charger_github(name)
    try:                                         # "auto"
        return _charger_local(name)
    except FileNotFoundError as err_local:
        try:
            return _charger_github(name)
        except Exception as err_github:
            raise RuntimeError(f"Instance {name} introuvable.\n  - local : {err_local}\n  - GitHub : {err_github}")


# ========================================================================
# 5.1. Données
# ========================================================================
def build_data(instance):
    """Range une instance vrplib dans un dictionnaire de listes Python simples.
    Nœud 0 = dépôt, nœuds 1..n = clients. Temps de trajet = distance."""
    coords = np.asarray(instance["node_coord"], dtype=float)
    nb = len(coords)
    dist = [[math.floor(10 * math.dist(coords[i], coords[j])) / 10   # convention DIMACS
             for j in range(nb)] for i in range(nb)]
    tw = instance["time_window"]
    service = instance.get("service_time", 0)
    if np.isscalar(service):                 # certains fichiers donnent une seule valeur
        service = [float(service)] * nb
    service = [float(x) for x in service]
    service[0] = 0.0                         # pas de service au dépôt
    return dict(
        n=nb - 1,                            # nombre de clients
        K=int(instance["vehicles"]),         # nombre de camions disponibles
        Q=float(instance["capacity"]),       # capacité d'un camion
        coords=coords,                       # (pour les graphiques uniquement)
        dist=dist,
        demande=[float(x) for x in instance["demand"]],
        debut=[float(x) for x in tw[:, 0]],  # e_i : ouverture de la fenêtre
        fin=[float(x) for x in tw[:, 1]],    # l_i : fermeture de la fenêtre
        service=service,                     # s_i : temps passé chez le client
    )


# ========================================================================
# 5.2. Coût et réalisabilité d'une tournée
# ========================================================================
# Une tournée = liste des clients dans l'ordre de visite, ex. [5, 12, 3].
# Le dépôt (0) n'est pas écrit : on part de 0 et on revient à 0 implicitement.
# Une solution = liste de tournées.

def cout_tournee(tournee, d):
    """Distance totale parcourue : 0 -> clients -> 0."""
    chemin = [0] + tournee + [0]
    return sum(d["dist"][chemin[k]][chemin[k + 1]] for k in range(len(chemin) - 1))


def cout_solution(solution, d):
    return sum(cout_tournee(t, d) for t in solution)


def tournee_realisable(tournee, d):
    """Simule le camion le long de la tournée et vérifie capacité + fenêtres de temps."""
    if sum(d["demande"][c] for c in tournee) > d["Q"]:
        return False                         # camion trop chargé
    heure, position = 0.0, 0                 # on part du dépôt à l'heure 0
    for c in tournee:
        heure += d["service"][position] + d["dist"][position][c]   # service précédent + trajet
        if heure > d["fin"][c]:
            return False                     # arrivé après la fermeture : interdit
        heure = max(heure, d["debut"][c])    # arrivé trop tôt : on attend
        position = c
    heure += d["service"][position] + d["dist"][position][0]       # retour au dépôt
    return heure <= d["fin"][0]              # le dépôt a aussi une heure de fermeture


# ========================================================================
# 5.3. Insertion au meilleur endroit
# ========================================================================
def positions_possibles(client, solution, d):
    """Liste de toutes les insertions réalisables : (surcoût, numéro de tournée, position).
    Numéro de tournée = len(solution) signifie « ouvrir un nouveau camion »."""
    D = d["dist"]
    options = []
    for r, tournee in enumerate(solution):
        if sum(d["demande"][c] for c in tournee) + d["demande"][client] > d["Q"]:
            continue                         # camion déjà trop plein : inutile de tester
        chemin = [0] + tournee + [0]
        for p in range(len(tournee) + 1):    # insertion entre chemin[p] et chemin[p+1]
            a, b = chemin[p], chemin[p + 1]
            surcout = D[a][client] + D[client][b] - D[a][b]
            if tournee_realisable(tournee[:p] + [client] + tournee[p:], d):
                options.append((surcout, r, p))
    if len(solution) < d["K"]:               # il reste un camion libre
        options.append((D[0][client] + D[client][0], len(solution), 0))
    return options


def inserer(client, r, p, solution):
    if r == len(solution):
        solution.append([client])            # nouveau camion
    else:
        solution[r].insert(p, client)


# ========================================================================
# 5.4. Méthodes de réparation
# ========================================================================
def reparation_gloutonne(solution, retires, d):
    """À chaque étape, insère le client dont l'insertion coûte le moins cher."""
    while retires:
        meilleur = None
        for c in retires:
            options = positions_possibles(c, solution, d)
            if not options:
                return False                 # un client ne rentre nulle part : échec
            o = min(options)
            if meilleur is None or o[0] < meilleur[0]:
                meilleur = (o[0], c, o[1], o[2])
        _, c, r, p = meilleur
        inserer(c, r, p, solution)
        retires.remove(c)
    return True


def reparation_regret(solution, retires, d):
    """Regret-2 : insère d'abord le client qui « perdrait le plus » s'il ne prenait pas
    sa meilleure tournée (écart entre sa 2e meilleure et sa meilleure tournée)."""
    while retires:
        choix = None
        for c in retires:
            options = positions_possibles(c, solution, d)
            if not options:
                return False
            par_tournee = {}                 # meilleur surcoût dans chaque tournée
            for surcout, r, p in options:
                if r not in par_tournee or surcout < par_tournee[r][0]:
                    par_tournee[r] = (surcout, r, p)
            tries = sorted(par_tournee.values())
            regret = tries[1][0] - tries[0][0] if len(tries) > 1 else float("inf")
            if choix is None or regret > choix[0]:
                choix = (regret, c, tries[0][1], tries[0][2])
        _, c, r, p = choix
        inserer(c, r, p, solution)
        retires.remove(c)
    return True


# ========================================================================
# 5.5. Méthodes de destruction
# ========================================================================
def retirer(clients, solution):
    for t in solution:
        t[:] = [c for c in t if c not in clients]
    solution[:] = [t for t in solution if t]  # on supprime les camions vides


def destruction_aleatoire(solution, q, d):
    """Retire q clients au hasard : diversifie la recherche."""
    tous = [c for t in solution for c in t]
    clients = random.sample(tous, q)
    retirer(clients, solution)
    return clients


def destruction_pire(solution, q, d):
    """Retire des clients qui provoquent les plus gros détours."""
    D = d["dist"]
    gains = []
    for t in solution:
        chemin = [0] + t + [0]
        for k in range(1, len(chemin) - 1):
            a, c, b = chemin[k - 1], chemin[k], chemin[k + 1]
            gains.append((D[a][c] + D[c][b] - D[a][b], c))
    gains.sort(reverse=True)
    candidats = [c for _, c in gains[:2 * q]]  # un peu de hasard : on pioche parmi les 2q pires
    clients = random.sample(candidats, q)
    retirer(clients, solution)
    return clients


def destruction_voisins(solution, q, d):
    """Retire un client au hasard + ses q-1 plus proches voisins : ils pourront être
    réorganisés ensemble (idée de Shaw, en version simplifiée)."""
    tous = [c for t in solution for c in t]
    graine = random.choice(tous)
    tous.sort(key=lambda c: d["dist"][graine][c])
    clients = tous[:q]                       # la graine elle-même est à distance 0
    retirer(clients, solution)
    return clients


# ========================================================================
# 5.6. Choix adaptatif des méthodes
# ========================================================================
def choisir(poids):
    """Roulette : une méthode est tirée avec une probabilité proportionnelle à son poids."""
    return random.choices(range(len(poids)), weights=poids)[0]


def mettre_a_jour(poids, scores, utilisations, reaction):
    """Fin d'un segment : chaque poids se rapproche du score moyen obtenu par la méthode."""
    for i in range(len(poids)):
        if utilisations[i] > 0:
            poids[i] = (1 - reaction) * poids[i] + reaction * scores[i] / utilisations[i]
        scores[i], utilisations[i] = 0, 0


# ========================================================================
# 5.7. Boucle principale (ALNS + recuit simulé)
# ========================================================================
DESTRUCTIONS = [destruction_aleatoire, destruction_pire, destruction_voisins]
REPARATIONS = [reparation_gloutonne, reparation_regret]
NOMS_D = ["aleatoire", "pire", "voisins"]
NOMS_R = ["gloutonne", "regret"]


def alns(d, iterations=2000, q_min=5, q_max=20, depart_pire=0.05, reaction=0.1,
         segment=100, graine=0):
    """ALNS + recuit simulé. Renvoie un dictionnaire de résultats."""
    random.seed(graine)
    t_debut = time.perf_counter()
    q_min, q_max = min(q_min, d["n"]), min(q_max, d["n"])

    # 1) Solution initiale : réparation gloutonne à partir d'une solution vide
    solution = []
    if not reparation_gloutonne(solution, list(range(1, d["n"] + 1)), d):
        raise ValueError("Pas de solution initiale réalisable avec K camions")
    cout = cout_solution(solution, d)
    cout_initial = cout
    meilleure, meilleur_cout = [t[:] for t in solution], cout

    # 2) Recuit : T0 tel qu'une solution `depart_pire` (5 %) plus chère soit acceptée 1 fois sur 2
    T = depart_pire * cout / math.log(2)
    alpha = (1 / 1000) ** (1 / iterations)   # T finit à T0 / 1000

    poids_d, poids_r = [1.0] * 3, [1.0] * 2
    scores_d, scores_r, util_d, util_r = [0] * 3, [0] * 2, [0] * 3, [0] * 2
    records_d, total_d = [0] * 3, [0] * 3    # statistiques pour l'étude des opérateurs
    hist_courant, hist_meilleur = [], []

    for it in range(1, iterations + 1):
        # 3) Choix des méthodes et de q
        i_d, i_r = choisir(poids_d), choisir(poids_r)
        q = random.randint(q_min, q_max)

        # 4) Détruire puis réparer une COPIE de la solution courante
        candidate = [t[:] for t in solution]
        retires = DESTRUCTIONS[i_d](candidate, q, d)
        ok = REPARATIONS[i_r](candidate, retires, d)

        # 5) Acceptation (recuit simulé) et points
        points = 0
        if ok:
            c = cout_solution(candidate, d)
            delta = c - cout
            if c < meilleur_cout - 1e-9:         # nouveau record
                meilleure, meilleur_cout = [t[:] for t in candidate], c
                points = 33
                records_d[i_d] += 1
            elif delta < -1e-9:                  # mieux que la solution courante
                points = 9
            if delta < 0 or random.random() < math.exp(-delta / T):
                solution, cout = candidate, c
                if points == 0 and delta > 1e-9:
                    points = 13                  # acceptée bien que pire

        # 6) Mise à jour des scores, puis des poids tous les `segment` itérations
        scores_d[i_d] += points; util_d[i_d] += 1; total_d[i_d] += 1
        scores_r[i_r] += points; util_r[i_r] += 1
        if it % segment == 0:
            mettre_a_jour(poids_d, scores_d, util_d, reaction)
            mettre_a_jour(poids_r, scores_r, util_r, reaction)
        T *= alpha
        hist_courant.append(cout)
        hist_meilleur.append(meilleur_cout)

    return dict(tournees=meilleure, cout=meilleur_cout, cout_initial=cout_initial,
                temps=time.perf_counter() - t_debut, hist_courant=hist_courant,
                hist_meilleur=hist_meilleur, poids_d=poids_d, poids_r=poids_r,
                records_d=records_d, total_d=total_d)


# ========================================================================
# 5.8. Vérificateur
# ========================================================================
def check_solution(d, tournees):
    """Vérification indépendante : chaque client une fois, flotte, capacité, fenêtres.
    Renvoie le coût recalculé."""
    visites = sorted(c for t in tournees for c in t)
    assert visites == list(range(1, d["n"] + 1)), "chaque client doit être visité une fois"
    assert len(tournees) <= d["K"], "trop de camions"
    total = 0.0
    for t in tournees:                       # recalcul complet, sans réutiliser le code du solveur
        assert sum(d["demande"][c] for c in t) <= d["Q"], f"capacité dépassée : {t}"
        heure, prec = d["debut"][0], 0
        for c in list(t) + [0]:
            heure = max(d["debut"][c], heure + d["service"][prec] + d["dist"][prec][c])
            assert heure <= d["fin"][c], f"fenêtre violée au nœud {c}"
            total += d["dist"][prec][c]
            prec = c
    return total


# ========================================================================
# 5.9. Cas de test
# ========================================================================
def toy_instance():
    """Dépôt au centre, 4 clients aux 4 coins d'un carré de côté 20. Q = 20, chaque demande = 10.
    Les fenêtres imposent l'ordre : client 1 tôt, client 3 tard."""
    return dict(node_coord=np.array([[0, 0], [10, 10], [-10, 10], [-10, -10], [10, -10]]),
                demand=np.array([0, 10, 10, 10, 10]), capacity=20, vehicles=4,
                time_window=np.array([[0, 1000], [0, 20], [0, 1000], [200, 300], [0, 1000]]),
                service_time=5)


def test_cas_simple():
    """Test sur une petite instance dont on connaît l'optimum (section 5.9)."""
    toy = build_data(toy_instance())
    res = alns(toy, iterations=200, graine=0)
    cost = check_solution(toy, res["tournees"])
    print("tournées :", res["tournees"], "| coût :", round(cost, 1))

    # 1) capacité 20 et demandes de 10 => 2 tournées de 2 clients
    assert len(res["tournees"]) == 2 and all(len(t) == 2 for t in res["tournees"])
    # 2) optimum = 2 tournées (dépôt -> coin -> coin adjacent -> dépôt) = 2 * (14.1 + 20 + 14.1)
    assert abs(cost - 2 * (14.1 + 20 + 14.1)) < 1e-6
    # 3) tests négatifs : le vérificateur doit rejeter une solution invalide
    for mauvaise, msg in [([[3, 1], [2, 4]], "fenêtre"), ([[1, 2, 3], [4]], "capacité")]:
        try:
            check_solution(toy, mauvaise)
            raise RuntimeError("le vérificateur aurait dû rejeter cette solution")
        except AssertionError as err:
            print(f"solution invalide ({msg}) correctement rejetée :", err)
    print("Tous les tests passent.")


# ========================================================================
# 5.10. Graphiques
# ========================================================================
def plot_solution(d, tournees, title, ax):
    xy = d["coords"]
    cmap = plt.get_cmap("tab20")
    for k, t in enumerate(tournees):
        chemin = [0] + list(t) + [0]
        ax.plot(xy[chemin, 0], xy[chemin, 1], "-", lw=1.3, color=cmap(k % 20))
    ax.scatter(xy[1:, 0], xy[1:, 1], s=10, c="k", zorder=3)
    ax.scatter(xy[0, 0], xy[0, 1], marker="s", s=90, c="red", zorder=4, label="dépôt")
    ax.set_title(title, fontsize=10)
    ax.set_aspect("equal")
    ax.legend(loc="upper right", fontsize=8)


# =====================================================================
# LANCEMENT
# =====================================================================
if LANCER_TEST_UNITAIRE:
    print("=== Test sur la petite instance (section 5.9) ===")
    test_cas_simple()
    print()

# Chargement de toutes les instances AVANT les calculs : une erreur de nom ou de chemin est détectée tout de suite
print(f"=== Chargement de {len(INSTANCES_A_LANCER)} instance(s) (source : {SOURCE_INSTANCES}) ===")
donnees = {}
for name in INSTANCES_A_LANCER:
    donnees[name] = load_instance(name)
    print(f"  {name} chargée")
print()

recap = []
for name in INSTANCES_A_LANCER:
    inst, bks = donnees[name]
    d = build_data(inst)
    print(f"=== {name} : {d['n']} clients, {NB_ITERATIONS} itérations (calcul en cours...) ===")
    res = alns(d, iterations=NB_ITERATIONS, graine=GRAINE)
    cost = check_solution(d, res["tournees"])
    gap = 100 * (cost - bks["cost"]) / bks["cost"]
    print(f"{name}: coût = {cost:.1f} | BKS = {bks['cost']} | gap = {gap:.2f} % | "
          f"camions = {len(res['tournees'])} (BKS : {len(bks['routes'])}) | "
          f"temps = {res['temps']:.0f} s | solution initiale = {res['cout_initial']:.1f}")
    recap.append((name, res["cout_initial"], cost, bks["cost"], gap, len(res["tournees"]), len(bks["routes"]), res["temps"]))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
    plot_solution(d, res["tournees"], f"{name} – ALNS : {cost:.1f} (gap {gap:.2f} %)", axes[0])
    ax = axes[1]
    ax.plot(res["hist_courant"], lw=0.6, alpha=0.7, label="solution courante")
    ax.plot(res["hist_meilleur"], lw=1.8, label="meilleure solution")
    ax.axhline(bks["cost"], color="k", ls="--", lw=1, label="BKS")
    ax.set_xlabel("itération")
    ax.set_ylabel("distance totale")
    ax.set_title(f"{name} – convergence", fontsize=10)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print()

print("=== Récapitulatif ===")
print(f"{'instance':<9}{'initiale':>10}{'ALNS':>10}{'BKS':>10}{'gap %':>8}{'camions':>9}{'(BKS)':>7}{'temps s':>9}")
for nom, ini, c, b, g, k, kb, t in recap:
    print(f"{nom:<9}{ini:>10.1f}{c:>10.1f}{b:>10.1f}{g:>8.2f}{k:>9d}{kb:>7d}{t:>9.0f}")
